# 📉 PredictiChurn – Training Pipeline

End-to-end notebook covering:
1. Data loading & cleaning
2. Feature engineering & encoding
3. Scaling + saving the preprocessor
4. SMOTE balancing
5. XGBoost training & evaluation
6. SHAP explainability
7. Exporting artefacts to `backend/`

> **Dataset**: [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
> Place the downloaded `WA_Fn-UseC_-Telco-Customer-Churn.csv` inside `data/`.

In [ ]:
# ─── 0. Imports ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

# ── Sklearn ────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Imbalanced-learn ──────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── SHAP ──────────────────────────────────────────────────────────────────────
import shap
shap.initjs()

print('All imports OK ✅')

---
## Phase 2 – Data Cleaning & Preprocessing

In [ ]:
# ─── 1. Load dataset ───────────────────────────────────────────────────────────
DATA_PATH = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

if not DATA_PATH.exists():
    print(
        f'⚠️  Dataset not found at {DATA_PATH}.\n'
        'Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n'
        'Place the CSV in the data/ folder and re-run this cell.'
    )
else:
    df = pd.read_csv(DATA_PATH)
    print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
    df.head()

In [ ]:
# ─── 2. Initial exploration ────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Target Distribution ===')
print(df['Churn'].value_counts())

In [ ]:
# ─── 3. Cleaning ──────────────────────────────────────────────────────────────
# Drop useless identifier column
df.drop(columns=['customerID'], inplace=True)

# TotalCharges arrives as object – convert; blank strings → 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)

# Target variable: Yes → 1, No → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Cleaning done ✅')
print(df[['TotalCharges', 'Churn']].dtypes)
print(df['Churn'].value_counts())

In [ ]:
# ─── 4. Feature / target split ────────────────────────────────────────────────
X = df.drop(columns=['Churn'])
y = df['Churn']

# Identify column types
NUMERIC_COLS  = ['tenure', 'MonthlyCharges', 'TotalCharges']
BINARY_COLS   = ['SeniorCitizen']          # already 0/1 int
CAT_COLS      = [
    c for c in X.columns
    if c not in NUMERIC_COLS + BINARY_COLS
]

print('Numeric  :', NUMERIC_COLS)
print('Binary   :', BINARY_COLS)
print('Categoric:', CAT_COLS)

In [ ]:
# ─── 5. Build preprocessor ────────────────────────────────────────────────────
#
# We use a ColumnTransformer that:
#   • Scales  numeric columns with StandardScaler
#   • Passes  binary column as-is
#   • Encodes categorical columns with OHE (drop='first' to avoid multicollinearity)
#
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    drop='first',        # drop first level → avoids dummy trap
    sparse_output=False, # return dense array
    handle_unknown='ignore',
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_COLS),
        ('bin', 'passthrough',       BINARY_COLS),
        ('cat', categorical_transformer, CAT_COLS),
    ],
    verbose_feature_names_out=False,  # cleaner names
)

print('Preprocessor built ✅')

In [ ]:
# ─── 6. Train / test split ────────────────────────────────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train : {X_train_raw.shape[0]:,} rows')
print(f'Test  : {X_test_raw.shape[0]:,} rows')
print(f'Train churn rate: {y_train.mean():.1%}')
print(f'Test  churn rate: {y_test.mean():.1%}')

In [ ]:
# ─── 7. Fit preprocessor on train; transform both splits ──────────────────────
X_train = preprocessor.fit_transform(X_train_raw)
X_test  = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out().tolist()
print(f'Feature count after encoding: {len(feature_names)}')
print('First 10 features:', feature_names[:10])

In [ ]:
# ─── 8. Save preprocessor ─────────────────────────────────────────────────────
PREPROCESSOR_PATH = Path('../backend/preprocessor.pkl')
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f'preprocessor.pkl saved → {PREPROCESSOR_PATH} ✅')

---
## Phase 3 – Handling Imbalance & Model Training

In [ ]:
# ─── 9. Class balance visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].pie(
    y_train.value_counts(),
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    colors=['#3b82f6', '#ef4444'],
)
axes[0].set_title('Before SMOTE')

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

axes[1].pie(
    pd.Series(y_train_sm).value_counts(),
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    colors=['#3b82f6', '#ef4444'],
)
axes[1].set_title('After SMOTE')

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/class_balance.png', dpi=120)
plt.show()

print(f'SMOTE: {X_train.shape[0]:,} → {X_train_sm.shape[0]:,} training samples')
print(pd.Series(y_train_sm).value_counts())

In [ ]:
# ─── 10. XGBoost Training ─────────────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    min_child_weight=5,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

print('\nModel trained ✅')

In [ ]:
# ─── 11. Evaluation ───────────────────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC-AUC Score: {roc_auc:.4f}')

In [ ]:
# ─── 12. ROC Curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#ef4444', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve – PredictiChurn')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../data/roc_curve.png', dpi=120)
plt.show()

In [ ]:
# ─── 13. Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# ─── 14. Cross-validation (Recall focus) ─────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Re-build on full pre-processed training data (without SMOTE in CV for speed)
recall_scores = cross_val_score(model, X_train_sm, y_train_sm,
                                 scoring='recall', cv=cv, n_jobs=-1)
roc_scores    = cross_val_score(model, X_train_sm, y_train_sm,
                                 scoring='roc_auc', cv=cv, n_jobs=-1)

print(f'CV Recall  : {recall_scores.mean():.4f} ± {recall_scores.std():.4f}')
print(f'CV ROC-AUC : {roc_scores.mean():.4f}  ± {roc_scores.std():.4f}')

In [ ]:
# ─── 15. Save model ───────────────────────────────────────────────────────────
MODEL_PATH = Path('../backend/model.pkl')
joblib.dump(model, MODEL_PATH)
print(f'model.pkl saved → {MODEL_PATH} ✅')

---
## Phase 4 – Model Explainability (SHAP)

In [ ]:
# ─── 16. SHAP Explainer ───────────────────────────────────────────────────────
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the entire test set
shap_values = explainer.shap_values(X_test)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# ─── 17. Global feature importance (beeswarm) ─────────────────────────────────
X_test_df = pd.DataFrame(X_test, columns=feature_names)

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test_df,
    plot_type='beeswarm',
    show=False,
)
plt.title('SHAP Beeswarm – Global Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig('../data/shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 18. Single-row waterfall plot ────────────────────────────────────────────
# Pick the first churner in the test set for demonstration
churner_idx = int(np.where(y_test.values == 1)[0][0])
print(f'Demonstrating on test index {churner_idx}')
print(f'Predicted prob : {y_pred_prob[churner_idx]:.4f}')
print(f'Actual label   : {y_test.iloc[churner_idx]}')

# Extract top-3 driving features
sv_row   = shap_values[churner_idx]
abs_sv   = np.abs(sv_row)
top3_idx = np.argsort(abs_sv)[::-1][:3]

print('\n--- Top 3 SHAP Factors ---')
for rank, idx in enumerate(top3_idx, start=1):
    direction = 'increases' if sv_row[idx] > 0 else 'decreases'
    print(f'  #{rank} {feature_names[idx]:40s}  SHAP={sv_row[idx]:+.4f}  ({direction} churn risk)')

In [ ]:
# ─── 19. Waterfall chart ──────────────────────────────────────────────────────
explanation = shap.Explanation(
    values      = shap_values[churner_idx],
    base_values = explainer.expected_value,
    data        = X_test[churner_idx],
    feature_names = feature_names,
)

shap.plots.waterfall(explanation, max_display=10, show=False)
plt.title(f'SHAP Waterfall – Test Sample #{churner_idx}', fontsize=13)
plt.tight_layout()
plt.savefig('../data/shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print('Waterfall saved ✅')

In [ ]:
# ─── 20. Summary ──────────────────────────────────────────────────────────────
print('=== PredictiChurn – Training Complete ===')
print(f'  Model        → {MODEL_PATH}')
print(f'  Preprocessor → {PREPROCESSOR_PATH}')
print(f'  ROC-AUC      → {roc_auc:.4f}')
print(f'  SHAP plots   → data/*.png')
print('\nNext steps:')
print('  1. cd predictichurn')
print('  2. uvicorn backend.main:app --reload')
print('  3. streamlit run frontend/app.py')